# **VITALS PREPROCESSING**

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount="true")


Mounted at /content/drive


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/MIMIC_Extracted"
OUTPUT_PATH  = "/content/drive/MyDrive/MIMIC_Extracted/preprocessed_data"
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Dataset path:", DATASET_PATH)
print("Output path :", OUTPUT_PATH)

Dataset path: /content/drive/MyDrive/MIMIC_Extracted
Output path : /content/drive/MyDrive/MIMIC_Extracted/preprocessed_data


In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os, json, warnings

warnings.filterwarnings("ignore")

In [ ]:
VITAL_ITEMIDS = {
    "HeartRate": [211, 220045],
    "MeanBP": [456, 52, 6702, 220052],
    "RespRate": [618, 220210],
    "Temp": [676, 223761],
    "SpO2": [646, 220277]
}

VAR_NAMES = list(VITAL_ITEMIDS.keys())

itemid_to_var = {}
for var, ids in VITAL_ITEMIDS.items():
    for i in ids:
        itemid_to_var[i] = var

needed_items = set(itemid_to_var.keys())


In [ ]:
print("\n[STEP 1] Loading CHARTEVENTS...")
chartevents = pd.read_csv(os.path.join(DATASET_PATH, "CHARTEVENTS.csv"))
chartevents.columns = chartevents.columns.str.lower()

print("Total rows:", len(chartevents))

chartevents = chartevents[
    (chartevents["itemid"].isin(needed_items)) &
    (chartevents["valuenum"].notna())
]

chartevents["charttime"] = pd.to_datetime(chartevents["charttime"], errors="coerce")
chartevents = chartevents.dropna(subset=["charttime"])

print("After vital filter:", len(chartevents))


[STEP 1] Loading CHARTEVENTS...
Total rows: 9177734
After vital filter: 747994


In [ ]:
print("\n[STEP 2] Loading ICUSTAYS...")
icustays = pd.read_csv(os.path.join(DATASET_PATH, "ICUSTAYS.csv"))
icustays.columns = icustays.columns.str.lower()

icustays["intime"] = pd.to_datetime(icustays["intime"], errors="coerce")
icustays["outtime"] = pd.to_datetime(icustays["outtime"], errors="coerce")

icustays = icustays.dropna(subset=["intime", "outtime"])

print("ICU stays:", len(icustays))


[STEP 2] Loading ICUSTAYS...
ICU stays: 1000


In [ ]:
chartevents["var_name"] = chartevents["itemid"].map(itemid_to_var)


In [ ]:
print("\n[STEP 3] Selecting last 24h...")

vitals = chartevents.merge(
    icustays[["icustay_id", "intime", "outtime"]],
    on="icustay_id",
    how="inner"
)

vitals = vitals[
    (vitals["charttime"] >= vitals["intime"]) &
    (vitals["charttime"] <= vitals["outtime"])
]

vitals["window_start"] = vitals["outtime"] - pd.Timedelta(hours=24)

short = (vitals["outtime"] - vitals["intime"]).dt.total_seconds() < 24 * 3600
vitals.loc[short, "window_start"] = vitals.loc[short, "intime"]

vitals_24h = vitals[vitals["charttime"] >= vitals["window_start"]].copy()

print("Measurements:", len(vitals_24h))
print("ICU stays:", vitals_24h["icustay_id"].nunique())


[STEP 3] Selecting last 24h...
Measurements: 103538
ICU stays: 998


In [ ]:
print("\n[STEP 4] Computing medians...")
global_medians = vitals_24h.groupby("var_name")["valuenum"].median()




[STEP 4] Computing medians...


In [ ]:
HOURS = 24
TIME_GRID = np.arange(0, HOURS + 1)


def build_matrix(df):
    df = df.copy()
    window_start = df["window_start"].iloc[0]

    df["rel"] = (df["charttime"] - window_start).dt.total_seconds() / 3600
    df["rel"] = df["rel"].clip(0, HOURS)
    df["bin"] = df["rel"].astype(int)

    pivot = df.pivot_table(
        index="bin",
        columns="var_name",
        values="valuenum",
        aggfunc="mean"
    )

    pivot = pivot.reindex(TIME_GRID)

    for v in VAR_NAMES:
        if v not in pivot.columns:
            pivot[v] = np.nan

    pivot = pivot[VAR_NAMES]

    pivot = pivot.ffill().bfill()

    for c in pivot.columns:
        pivot[c] = pivot[c].fillna(global_medians.get(c, 0))

    return pivot.values.astype(np.float32)


In [ ]:
print("\n[STEP 5] Building tensors...")

X_list = []
ids = []

for icu_id, df in tqdm(vitals_24h.groupby("icustay_id")):
    mat = build_matrix(df)
    X_list.append(mat)
    ids.append(icu_id)

X = np.stack(X_list)
ids = np.array(ids)

print("Tensor shape:", X.shape)


[STEP 5] Building tensors...


100%|██████████| 998/998 [00:11<00:00, 88.41it/s] 

Tensor shape: (998, 25, 5)


In [ ]:
print("\n[STEP 6] Normalizing...")

mean = X.mean(axis=(0, 1), keepdims=True)
std = X.std(axis=(0, 1), keepdims=True) + 1e-6

X_norm = (X - mean) / std


[STEP 6] Normalizing...


In [ ]:
np.savez_compressed(
    os.path.join(OUTPUT_PATH, "vitals_24h_preprocessed.npz"),
    X=X_norm,
    ids=ids,
    mean=mean.squeeze(),
    std=std.squeeze(),
    var_names=np.array(VAR_NAMES)
)

print("Saved vitals_24h_preprocessed.npz")

Saved vitals_24h_preprocessed.npz


In [ ]:
print("\n[STEP 7] Saving CSV...")

rows = []
for i, icu_id in enumerate(ids):
    for t in range(HOURS + 1):
        row = {"icustay_id": int(icu_id), "hour": int(t)}
        for j, var in enumerate(VAR_NAMES):
            row[var] = float(X_norm[i, t, j])
        rows.append(row)

df_all = pd.DataFrame(rows)
df_all.to_csv(os.path.join(OUTPUT_PATH, "all_patients_vitals.csv"), index=False)

print("Saved all_patients_vitals.csv")
print("Shape:", df_all.shape)


print("\nPREPROCESSING COMPLETE")
print("Files:", os.listdir(OUTPUT_PATH))


[STEP 7] Saving CSV...
Saved all_patients_vitals.csv
Shape: (24950, 7)

PREPROCESSING COMPLETE
Files: ['vitals_24h_preprocessed.npz', 'all_patients_vitals.csv']


In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/MIMIC_Extracted/preprocessed_data/all_patients_vitals.csv"

df = pd.read_csv(path)

print("Total rows:", len(df))
print("Unique ICU stays:", df["icustay_id"].nunique())


Total rows: 24950
Unique ICU stays: 998


In [ ]:
icu = pd.read_csv("/content/drive/MyDrive/MIMIC_Extracted/ICUSTAYS.csv")

merged = df.merge(icu[["icustay_id", "subject_id"]], on="icustay_id")

print("Unique patients:", merged["subject_id"].nunique())

Unique patients: 988


# **CLINICAL NOTES PREPROCESSING**

In [ ]:
BASE_PATH = "/content/drive/MyDrive/MIMIC_Extracted"
DATA_PATH = BASE_PATH
OUTPUT_PATH = os.path.join(BASE_PATH, "notes_output")

os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Loading files...")

notes = pd.read_csv(os.path.join(DATA_PATH, "NOTEEVENTS.csv"), low_memory=False)
icustays = pd.read_csv(os.path.join(DATA_PATH, "ICUSTAYS.csv"))

notes.columns = notes.columns.str.lower()
icustays.columns = icustays.columns.str.lower()

print("Notes rows:", len(notes))


print("\nFiltering categories...")

notes["category"] = notes["category"].astype(str).str.strip().str.lower()

KEEP = [
    "physician",
    "discharge summary",
    "consult",
    "physician resident"
]

notes = notes[notes["category"].isin(KEEP)]

print("After filter:", len(notes))





Loading files...
Notes rows: 47575

Filtering categories...
After filter: 6506


In [ ]:
# ======================================================
# CLEAN TEXT
# ======================================================

print("\nCleaning text...")

def clean_text(t):
    t = str(t)

    # remove PHI
    t = re.sub(r"\[\*\*.*?\*\*\]", "", t)

    # remove weird characters
    t = re.sub(r"[^a-zA-Z0-9.,;:%()\-/ ]", " ", t)

    # remove extra spaces
    t = re.sub(r"\s+", " ", t)

    return t.lower().strip()


notes["clean_text"] = notes["text"].apply(clean_text)


# ======================================================
# FIX DATATYPES
# ======================================================

notes["hadm_id"] = pd.to_numeric(notes["hadm_id"], errors="coerce")
icustays["hadm_id"] = pd.to_numeric(icustays["hadm_id"], errors="coerce")

notes = notes.dropna(subset=["hadm_id"])
icustays = icustays.dropna(subset=["hadm_id"])


# ======================================================
# MAP TO ICU STAY
# ======================================================

print("\nMapping admissions → ICU stay...")

notes = notes.merge(
    icustays[["hadm_id", "icustay_id"]],
    on="hadm_id",
    how="inner"
)

print("After ICU mapping:", len(notes))


# ======================================================
# SORT BY TIME (IMPORTANT)
# ======================================================

if "charttime" in notes.columns:
    notes["charttime"] = pd.to_datetime(notes["charttime"], errors="coerce")
    notes = notes.sort_values(["icustay_id", "charttime"])


# ======================================================
# MERGE MULTIPLE NOTES PER ICU
# ======================================================

print("\nAggregating notes per ICU...")

icu_notes = (
    notes.groupby("icustay_id")["clean_text"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)


# ======================================================
# LENGTH CONTROL
# ======================================================

# Rough safe char limit for 512 tokens
MAX_CHARS = 3000

icu_notes["notes_text"] = icu_notes["clean_text"].str.slice(0, MAX_CHARS)
icu_notes = icu_notes.drop(columns=["clean_text"])


# ======================================================
# FINAL STATS
# ======================================================

print("\nFinal ICU stays with notes:", len(icu_notes))

out_file = os.path.join(OUTPUT_PATH, "notes_preprocessed.csv")
icu_notes.to_csv(out_file, index=False)

print("\nSaved:", out_file)


Cleaning text...

Mapping admissions → ICU stay...
After ICU mapping: 6541

Aggregating notes per ICU...

Final ICU stays with notes: 995

Saved: /content/drive/MyDrive/MIMIC_Extracted/notes_output/notes_preprocessed.csv
